In [8]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!apt-get install -y openjdk-11-jdk
!wget -q -O - https://github.com/SpencerPark/IJava/releases/download/v1.3.0/ijava-1.3.0.tar.gz | tar xzf -
!python3 install.py --sys-prefix


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 261 kB in 1s (178 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pa

In [ ]:
import csv
import os
import time
from datetime import date
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# Inicialize o WebDriver com modo headless
opcoes = webdriver.ChromeOptions()
opcoes.add_argument('--headless')
opcoes.add_argument('--no-sandbox')
opcoes.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opcoes)
driver.set_page_load_timeout(600)  # Aumentar o timeout para 10 minutos

# Função para navegar até uma página específica com retry
def navegar_para_pagina(url, tentativas=3):
    for tentativa in range(tentativas):
        try:
            driver.get(url)
            print(f"Página carregada: {url}")
            return True
        except TimeoutException:
            print(f"Timeout ao carregar a página: {url}, tentativa {tentativa + 1} de {tentativas}")
            tirar_screenshot(f"timeout_{url}_tentativa_{tentativa + 1}.png")
    return False

# Função para rolar até o final da página
def rolar_ate_final():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)  # Esperar 5 segundos para garantir o carregamento dos elementos

# Função para tirar uma captura de tela e salvar no Google Drive
def tirar_screenshot(nome_arquivo):
    screenshot_folder = '/content/drive/MyDrive/WebScraping/Joli/Screenshots'
    if not os.path.exists(screenshot_folder):
        os.makedirs(screenshot_folder)
    screenshot_path = os.path.join(screenshot_folder, nome_arquivo)
    driver.save_screenshot(screenshot_path)
    print(f"Captura de tela salva em: {screenshot_path}")

# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"A pasta {output_folder} foi criada.")

# URL da página de teste
url = "https://www.joli.com.br/material-de-construcao?page=1"

# Navegar para a página
if navegar_para_pagina(url):
    # Rolagem e espera para garantir carregamento dos produtos
    rolar_ate_final()

    try:
        WebDriverWait(driver, 300).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem'))
        )
        print("Elementos carregados na página.")
    except TimeoutException:
        print("Timeout ao esperar pelos elementos na página.")
        tirar_screenshot("timeout_elementos.png")

    # Extrair dados com BeautifulSoup
    page_source = driver.page_source
    soup = BeautifulSoup(page_source, 'html.parser')
    produtos_atual = soup.select('.vtex-search-result-3-x-galleryItem')

    # Adicione os produtos à lista
    produtos = []
    for produto in produtos_atual:
        try:
            # Coletar informações do produto
            nome_empresa = "Joli"
            data_coleta = date.today().strftime("%Y-%m-%d")
            codigo_produto = produto.select_one('.vtex-product-summary-2-x-brandName').text.strip() if produto.select_one('.vtex-product-summary-2-x-brandName') else "0"
            imagem_url = produto.select_one('.vtex-product-summary-2-x-image')['src'] if produto.select_one('.vtex-product-summary-2-x-image') else ""
            nome_produto = produto.select_one('.vtex-product-summary-2-x-productNameContainer').text.strip() if produto.select_one('.vtex-product-summary-2-x-productNameContainer') else ""
            moeda = produto.select_one('.joli-joli-theme-0-x-price').text.strip() if produto.select_one('.joli-joli-theme-0-x-price') else ""
            url_produto = produto.select_one('a')['href'] if produto.select_one('a') else ""

            # Montar a linha de dados do produto
            linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

            # Adicionar à lista de produtos
            produtos.append(linha_produto)
        except Exception as e:
            print(f"Erro ao extrair dados do produto: {e}")

    # Salvar os produtos em um arquivo CSV
    nome_arquivo = os.path.join(output_folder, 'produtos_teste.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])
        for produto in produtos:
            escritor_csv.writerow(produto)

    print(f"Produtos salvos em: {nome_arquivo}")

# Feche o driver
driver.quit()


Mounted at /content/drive
